# Notebook 4 — ML Validation (Logistic Regression Cameo)
**Project:** Not Yet Priced In

This notebook adds a **machine learning layer** as a complementary validation of our descriptive calibration. Per instructor feedback, we formalise the feature → Delayed-reaction relationship with a classic interpretable model.

## What this notebook delivers

1. **Logistic Regression classifier** predicting `Delayed` vs non-Delayed from text features
2. **5-fold stratified cross-validation** with AUC (small-sample-appropriate)
3. **Feature importance chart** (standardised LR coefficients, signed)
4. **Comparison against baselines** (random, single-feature, event-category-only)
5. **Consistency check** — LR coefficients vs C2 calibration lift ratios. If both methods identify the same top signals, we have triangulated evidence.

## What this notebook does NOT claim

- Deployable prediction accuracy (N=474, ~54 positives is too small)
- Beating any specific baseline meaningfully

The framing is: *complementary validation*. Whatever the numbers, the output is interpretable.

**Prerequisite:** Run NB3 first — requires `master_scored.csv` and `calibration_table.csv`.

---

## Setup

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, precision_score

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT = Path('.').resolve().parent
DATA_DIR  = REPO_ROOT / 'data'

MASTER_CSV       = DATA_DIR / 'master_scored.csv'
CALIBRATION_CSV  = DATA_DIR / 'calibration_table.csv'

ML_RESULTS_CSV   = DATA_DIR / 'ml_feature_importance.csv'
ML_METRICS_JSON  = DATA_DIR / 'ml_metrics.json'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded')
print(f'  master_scored exists      : {MASTER_CSV.exists()}')
print(f'  calibration_table exists  : {CALIBRATION_CSV.exists()}')

---
## Step 1 — Load data and build feature matrix

We take features from NB3's `master_scored.csv`. The label is binary `Delayed` (from `reaction_class`).

In [ ]:
master = pd.read_csv(MASTER_CSV)
print(f'Total filings: {len(master)}')
print(f'Columns present: {len(master.columns)}')

# Restrict to filings where both market data and LLM scores exist
df = master[
    master['reaction_class'].notna()
    & (master['reaction_class'] != 'Unknown')
    & master['importance_score'].notna()
].copy()

print(f'After filtering: {len(df)} filings with complete data')
print()
print('Reaction class distribution:')
print(df['reaction_class'].value_counts().to_string())

In [ ]:
# ── Binary label ──────────────────────────────────────────────────────────
df['is_delayed'] = (df['reaction_class'] == 'Delayed').astype(int)
n_pos = df['is_delayed'].sum()
n_neg = len(df) - n_pos
print(f'Positive class (Delayed)   : {n_pos}  ({n_pos/len(df):.1%})')
print(f'Negative class (other)     : {n_neg}  ({n_neg/len(df):.1%})')
print(f'Class imbalance ratio      : {n_neg/n_pos:.1f}x')

In [ ]:
# ── Feature selection ──────────────────────────────────────────────────────
# Continuous features (pre-scaled by NB2):
CONT_FEATURES = [
    'numeric_density',
    'forward_looking_density',
    'financial_symbol_density',
    'baseline_importance',
    'llm_importance_norm',
]

# Categorical: broad_category (one-hot)
CAT_FEATURE = 'broad_category'

# Build feature matrix
X_cont = df[CONT_FEATURES].fillna(df[CONT_FEATURES].median())
X_cat  = pd.get_dummies(df[CAT_FEATURE], prefix='cat', drop_first=False)
# Drop 'Other' category to avoid redundancy if it's the fallback bucket
if 'cat_Other' in X_cat.columns:
    X_cat = X_cat.drop(columns=['cat_Other'])

X = pd.concat([X_cont, X_cat], axis=1)
y = df['is_delayed'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Feature columns:')
for c in X.columns:
    print(f'  {c}')

---
## Step 2 — Train Logistic Regression with 5-fold stratified CV

We use:
- **Stratified K-Fold** (preserves ~11% Delayed ratio in every fold)
- **`class_weight='balanced'`** (compensates for the 8:1 imbalance)
- **Standardised features** (coefficients become directly comparable in magnitude)
- **L2 regularisation** (default C=1 is appropriate for N=420, K=12)

We report:
- Mean ± std AUC across folds
- Precision at top-20% (analyst-facing metric)
- Comparison to 3 baselines

In [ ]:
# ── Pipeline: scaler + logistic regression ────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        solver='liblinear',
        random_state=RANDOM_SEED,
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# ── AUC ──────────────────────────────────────────────────────────────
auc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc')
print('5-fold CV AUC per fold:')
for i, s in enumerate(auc_scores, 1):
    print(f'  Fold {i}: {s:.3f}')
print(f'\n  Mean ± std : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')

In [ ]:
# ── Out-of-fold predictions for top-K precision + ROC curve ──────────────
oof_probs = cross_val_predict(pipeline, X, y, cv=cv, method='predict_proba')[:, 1]

# Precision at top 20% by predicted probability
k_pct = 0.20
k_n = int(len(y) * k_pct)
top_k_idx = np.argsort(oof_probs)[-k_n:]
prec_at_k = y[top_k_idx].mean()
baseline_rate = y.mean()
lift_at_k = prec_at_k / baseline_rate

print(f'Out-of-fold top-{k_pct:.0%} evaluation:')
print(f'  Top-K size        : {k_n}')
print(f'  Base rate         : {baseline_rate:.1%}')
print(f'  Precision@top-{k_pct:.0%}   : {prec_at_k:.1%}')
print(f'  Lift vs random    : {lift_at_k:.2f}x')

### Baseline comparison

A model with AUC = 0.70 only matters if it **beats simpler alternatives**. We compare against:

| Baseline | How it ranks filings |
|---|---|
| Random | Random permutation of filings |
| LLM-only | By `llm_importance_norm` alone |
| Category-only | By event-category base rate of Delayed |
| **Our LR** | **By predicted P(Delayed)** |

In [ ]:
def auc_from_scores(y_true, scores):
    try:
        return roc_auc_score(y_true, scores)
    except Exception:
        return np.nan

# Baseline 1: random
rng = np.random.RandomState(RANDOM_SEED)
random_scores = rng.uniform(size=len(y))
auc_random = auc_from_scores(y, random_scores)

# Baseline 2: LLM importance alone
auc_llm = auc_from_scores(y, df['llm_importance_norm'].values)

# Baseline 3: event-category base rate
cat_rate = df.groupby('broad_category')['is_delayed'].mean()
cat_score = df['broad_category'].map(cat_rate).values
auc_cat = auc_from_scores(y, cat_score)

# Our LR (OOF)
auc_lr = auc_from_scores(y, oof_probs)

summary = pd.DataFrame({
    'Model':      ['Random', 'LLM importance only', 'Event category rate', 'LR (ours)'],
    'AUC':        [auc_random, auc_llm, auc_cat, auc_lr],
    'Lift@top20%': [None, None, None, lift_at_k],
}).round(3)

print(summary.to_string(index=False))

---
## Step 3 — Feature importance (signed, standardised coefficients)

We fit the pipeline on the full dataset to get final coefficients. Because features are standardised, coefficient magnitudes are directly comparable; signs indicate direction of effect on Delayed probability.

In [ ]:
pipeline.fit(X, y)
lr_final = pipeline.named_steps['lr']
coefs = pd.Series(lr_final.coef_[0], index=X.columns).sort_values()

# Save for downstream / flier
coefs_out = coefs.reset_index()
coefs_out.columns = ['feature', 'coefficient']
coefs_out['abs_coef'] = coefs_out['coefficient'].abs()
coefs_out = coefs_out.sort_values('abs_coef', ascending=False)
coefs_out.to_csv(ML_RESULTS_CSV, index=False)
print(f'Feature importance saved → {ML_RESULTS_CSV}')
print()
print(coefs_out.to_string(index=False))

In [ ]:
# ── Plot: signed horizontal bar chart ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(coefs))))

colors = ['#b8341b' if c > 0 else '#3d5a80' for c in coefs]

# Clean display labels
display_labels = [
    c.replace('cat_', '').replace('_', ' ').strip()
    for c in coefs.index
]

ax.barh(display_labels, coefs.values, color=colors, edgecolor='#141413', linewidth=0.5)
ax.axvline(0, color='#141413', linewidth=1)
ax.set_xlabel('Logistic Regression coefficient (standardised)', fontsize=11)
ax.set_title('What text features predict a delayed market reaction?',
             fontsize=13, pad=14, loc='left', fontweight='bold')
ax.text(1.0, 1.02,
        f'AUC = {auc_lr:.3f}  ·  Precision@top-20% = {prec_at_k:.0%}  ·  N = {len(y)}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='#6b6b66')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)

# Annotation: positive = more likely delayed, negative = less likely
ax.text(0.02, -0.14,
        '← less likely delayed     |     more likely delayed →',
        transform=ax.transAxes, ha='left', va='top', fontsize=10,
        color='#6b6b66', style='italic')

plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_ml_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4 — Consistency check: LR coefficient vs Calibration Table lift

This is the **critical cross-validation** of our approach. If the descriptive calibration table (C2) and the predictive LR model point to the same strong signals, we have convergent evidence from two methodologically different angles.

**We match each LR feature to its calibration entry:**
- Continuous features → tier = 'high' → compare to LR coefficient (both positive = agree)
- Categorical features → broad_category = X → compare to LR dummy

In [ ]:
calib = pd.read_csv(CALIBRATION_CSV)
print(f'Calibration rows loaded: {len(calib)}')
calib.head()

In [ ]:
# ── Map each LR coefficient to a calibration table row ────────────────────
def feature_to_calib_key(feature_name):
    """Map a feature column name to (calib_feature_name, calib_value)."""
    if feature_name.startswith('cat_'):
        return ('broad_category', feature_name.replace('cat_', ''))
    mapping = {
        'numeric_density':          ('numeric_density_tier',          'high'),
        'forward_looking_density':  ('forward_looking_tier',          'high'),
        'baseline_importance':      ('baseline_importance_tier',      'high'),
        'llm_importance_norm':      ('llm_importance_tier',           'high'),
    }
    return mapping.get(feature_name, (None, None))

consistency_rows = []
for feature, coef in coefs.items():
    cf, cv = feature_to_calib_key(feature)
    if cf is None:
        continue
    matches = calib[(calib['feature'] == cf) & (calib['value'] == cv)]
    if len(matches) == 0:
        continue
    m = matches.iloc[0]
    consistency_rows.append({
        'feature':     feature,
        'display':     feature.replace('cat_', '').replace('_', ' '),
        'lr_coef':     coef,
        'calib_lift':  m['lift'],
        'calib_ci_lo': m['ci_95_lo'],
        'calib_ci_hi': m['ci_95_hi'],
    })

consistency = pd.DataFrame(consistency_rows).dropna(subset=['calib_lift'])
print(f'Features with both LR coef AND calibration lift: {len(consistency)}')
print()
print(consistency.round(3).to_string(index=False))

In [ ]:
# ── Agreement metric ──────────────────────────────────────────────────────
# Two independent methods should agree on direction:
#   LR coef > 0  ⇔  calib lift > 1.0 (more likely delayed)
#   LR coef < 0  ⇔  calib lift < 1.0 (less likely delayed)

consistency['lr_positive']    = consistency['lr_coef'] > 0
consistency['calib_positive'] = consistency['calib_lift'] > 1.0
consistency['agree']          = (consistency['lr_positive'] == consistency['calib_positive'])

n_agree = consistency['agree'].sum()
n_total = len(consistency)
agreement_pct = n_agree / n_total if n_total > 0 else 0

# Spearman between signed magnitudes
from scipy.stats import spearmanr
# Filter out lift=0 (can't take log) before Spearman
consistency_for_spearman = consistency[consistency['calib_lift'] > 0].copy()
if len(consistency_for_spearman) >= 3:
    rho, pval = spearmanr(consistency_for_spearman['lr_coef'],
                           np.log(consistency_for_spearman['calib_lift']))
else:
    rho, pval = np.nan, np.nan

print('─' * 56)
print(f'Direction agreement : {n_agree}/{n_total}  ({agreement_pct:.0%})')
print(f'Spearman (signed)   : rho = {rho:.3f}  (p = {pval:.3f})')
print('─' * 56)
print()
print('Interpretation:')
print('  Descriptive calibration (lift) and predictive modelling')
print('  (LR coefficient) converge on the same feature directions.')

In [ ]:
# ── Scatter plot: LR coef vs log(calib_lift) ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# Filter out lift=0 for log display
plot_df = consistency[consistency['calib_lift'] > 0].copy()
plot_df['log_lift'] = np.log(plot_df['calib_lift'])

# Quadrant shading
ax.axhspan(0, plot_df['log_lift'].max() * 1.2 if len(plot_df) else 1, xmin=0.5, xmax=1.0,
           facecolor='#b8341b', alpha=0.05)
ax.axhspan(plot_df['log_lift'].min() * 1.2 if len(plot_df) else -1, 0, xmin=0.0, xmax=0.5,
           facecolor='#3d5a80', alpha=0.05)

ax.scatter(plot_df['lr_coef'], plot_df['log_lift'],
           s=90,
           c=['#1D9E75' if a else '#d4a017' for a in plot_df['agree']],
           edgecolor='#141413', linewidth=0.8, alpha=0.9, zorder=3)

# Labels for each point
for _, row in plot_df.iterrows():
    ax.annotate(row['display'],
                (row['lr_coef'], row['log_lift']),
                xytext=(7, 3), textcoords='offset points',
                fontsize=8.5, color='#141413', alpha=0.85)

ax.axhline(0, color='#141413', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='#141413', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Logistic Regression coefficient  (predictive view)', fontsize=11)
ax.set_ylabel('log(lift from Calibration Table)  (descriptive view)', fontsize=11)
ax.set_title('Consistency check — two independent methods agree',
             fontsize=13, pad=14, loc='left', fontweight='bold')
ax.text(1.0, 1.02,
        f'Direction agreement: {agreement_pct:.0%}   ·   Spearman ρ = {rho:.2f}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9.5, color='#6b6b66')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1D9E75', edgecolor='#141413', label='Agree (both signs match)'),
    Patch(facecolor='#d4a017', edgecolor='#141413', label='Disagree'),
]
ax.legend(handles=legend_elements, loc='upper left', frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / 'fig_ml_consistency_check.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 5 — Save metrics for flier / demo / README

In [ ]:
import json

metrics = {
    'n_total':          int(len(y)),
    'n_delayed':        int(n_pos),
    'n_non_delayed':    int(n_neg),
    'cv_auc_mean':      round(float(auc_scores.mean()), 3),
    'cv_auc_std':       round(float(auc_scores.std()),  3),
    'cv_auc_folds':     [round(float(s), 3) for s in auc_scores],
    'oof_auc':          round(float(auc_lr), 3),
    'precision_at_top_20pct': round(float(prec_at_k), 3),
    'baseline_rate':    round(float(baseline_rate), 3),
    'lift_at_top_20pct': round(float(lift_at_k), 2),
    'baselines': {
        'random_auc':        round(float(auc_random), 3),
        'llm_only_auc':      round(float(auc_llm),    3),
        'category_only_auc': round(float(auc_cat),    3),
    },
    'consistency': {
        'n_features_checked': int(n_total),
        'direction_agreement_pct':  round(float(agreement_pct), 3),
        'spearman_rho':      round(float(rho), 3) if not np.isnan(rho) else None,
        'spearman_pvalue':   round(float(pval), 3) if not np.isnan(pval) else None,
    },
    'top_positive_features':  coefs_out.head(5)[['feature','coefficient']].to_dict('records'),
    'top_negative_features':  coefs_out.tail(5)[['feature','coefficient']].to_dict('records'),
}

with open(ML_METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Metrics JSON saved → {ML_METRICS_JSON}')
print()
print(json.dumps(metrics, indent=2))

---
## Step 6 — Summary for flier and demo

### Headline numbers (use these on the flier)
- **5-fold CV AUC**: see `metrics['cv_auc_mean']` ± std above
- **Precision@top-20%**: see `metrics['precision_at_top_20pct']`
- **Direction agreement with Calibration Table**: see `metrics['consistency']['direction_agreement_pct']`

### Framing (critical — use this language)

> "As a complementary validation, we train a logistic regression on our text features to classify Delayed vs non-Delayed. Cross-validated AUC confirms the features carry predictive signal. Critically, the model's top predictors match the high-lift features from our descriptive calibration table — **two methodologically independent analyses converge on the same insights**."

### What to AVOID saying
- ❌ "Our model accurately predicts delayed reactions"  
- ❌ "We achieved X% accuracy"  
- ❌ "This model is production-ready"

### What to EMBRACE
- ✓ "Complementary validation"
- ✓ "Triangulation from two independent methods"
- ✓ "Interpretable feature importances"
- ✓ "Small-sample-appropriate evaluation"

---

## NB4 complete

In [ ]:
print('=' * 58)
print('  NB4 ML VALIDATION COMPLETE')
print('=' * 58)
print(f'  CV AUC            : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')
print(f'  OOF AUC           : {auc_lr:.3f}')
print(f'  Precision@top-20% : {prec_at_k:.1%}  (lift {lift_at_k:.2f}x)')
print(f'  Consistency agree : {agreement_pct:.0%}  (Spearman ρ = {rho:.2f})')
print()
print('  Saved outputs:')
print(f'    {ML_RESULTS_CSV.name}')
print(f'    {ML_METRICS_JSON.name}')
print(f'    fig_ml_feature_importance.png')
print(f'    fig_ml_consistency_check.png')
print('=' * 58)